# Preliminary rule-based CPC screening — smart glasses
**What this file does:** it selects up to 50 DOCDB families per listed CPC prefix in a fixed time window, retrieves one English representative text per family, and searches that text for predefined phrases. It reports text coverage, phrase matches, audio/interaction flags, overlapping prefixes and heuristic recommendations. SQL, raw text, rule source and settings are saved.

**What it does not do:** it does not discover keywords, apply TF-IDF, verify official CPC definitions, measure precision/recall, prove relevance, or modify the extraction scope. The phrases were manually defined before seeing the data. A missing phrase is not proof of irrelevance, and a phrase match can be a false positive. Recommendations must not automatically remove CPC codes.

**Place in the project:** use `sector_keywords.ipynb` first for tutorial-aligned candidate collection, family/date/language checks, and descriptive keyword extraction. This notebook is an optional supplementary diagnostic. TF-IDF is an extension to Tutorials 0–1, not a replacement for relevance validation. BM25/SBERT retrieval in Tutorial 0 is a separate possible follow-up.

Run on TIP. TEST is only for development; PROD is required for final sector conclusions. Record the PATSTAT release. The default 2000–2022 window is a configurable analysis choice, not a guarantee of complete data. Sampling is deterministic for a fixed snapshot. Samples overlap across prefixes, so counts cannot be added to obtain sector size.

In [ ]:
from pathlib import Path
import sys

# Works when launched from the repository root or project directory.
PROJECT_DIR = Path.cwd() if Path("cpc_validation.py").exists() else Path.cwd() / "project"
if not (PROJECT_DIR / "cpc_validation.py").exists():
    raise RuntimeError("Open this notebook from the repository root or project directory")
sys.path.insert(0, str(PROJECT_DIR.resolve()))

from epo.tipdata.patstat import PatstatClient
from cpc_validation import run_validation
import pandas as pd

%run "$PROJECT_DIR/key_words.ipynb"

ENV = "TEST"
DATASET_RELEASE = ""  # Required: copy the PATSTAT release identifier from TIP.
YEAR_MIN, YEAR_MAX = 2000, 2022
SAMPLE_SIZE = 50
SEED = 42


In [ ]:
if not DATASET_RELEASE.strip():
    raise ValueError("Set DATASET_RELEASE to the PATSTAT release identifier shown in TIP")

patstat = PatstatClient(env=ENV)
summaries, evidence, output_path = run_validation(
    patstat, CPC_CODES, env=ENV, dataset_release=DATASET_RELEASE,
    year_min=YEAR_MIN, year_max=YEAR_MAX,
    sample_size=SAMPLE_SIZE, seed=SEED,
    output_dir=PROJECT_DIR / "outputs" / "cpc_validation",
)
summary_df = pd.DataFrame(summaries)
display(summary_df)
print(f"Saved outputs: {output_path.resolve()}")


## Reading the output
- `KEEP_CANDIDATE`: at least 70% of readable sampled families meet the phrase rules.
- `REVIEW_MIXED_SCOPE`: 30–70% meet the rules.
- `CONSIDER_NARROWING_WITH_TEXT`: fewer than 30% meet the rules; this does not prove irrelevance.
- `INSUFFICIENT_TEXT_OR_SAMPLE`: fewer than 20 readable families or English text coverage below 80%.
- `NO_DATA_IN_SELECTED_ENVIRONMENT`: no eligible families in this environment and time window.
- `covered_by_listed_prefix`: a shorter prefix already includes this code under the project's current string-prefix search. This is not a verification of the official CPC hierarchy.

`cpc_summary.csv` contains recommendations. `family_evidence.csv` contains sampled texts, themes and matched phrases. The queries fetch family members beyond those carrying the matched CPC, preferring an English abstract, then an English title, then earliest filing date and smallest application ID. Missing English text stays in the coverage denominator.

The extraction notebook and CPC list are unchanged. Decide on dataset scope separately before applying screening recommendations to extraction.